In [ ]:
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch
from torch.utils.data import DataLoader
from torch import nn

In [3]:
training_data = datasets.MNIST(root = ".", train = True, download = True, transform=  ToTensor())
test_data = datasets.MNIST(root = ".", train = False, download = True, transform = ToTensor())
img, label = training_data[0]

100%|██████████| 9.91M/9.91M [00:03<00:00, 2.82MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 327kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.61MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.85MB/s]


In [4]:
loaded_train = DataLoader(training_data, batch_size = 64, shuffle = True)
loaded_test = DataLoader(test_data, batch_size = 64, shuffle = True)

In [5]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout, kernel_size, stride=1, padding=0):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, kernel_size, stride, padding)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d((2, 2))
    
    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = self.pool(x)
        return x

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.cb1 = ConvBlock(1, 8, (5, 5), padding=2)
        self.cb2 = ConvBlock(8, 8, (5, 5), padding=2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(392, 10)
        self.softmax = nn.Softmax(dim=0)     

    def forward(self, x):
        x = self.cb1(x)
        x = self.cb2(x)
        x = self.flatten(x)
        x = self.fc(x)
        x = self.softmax(x)
        return x

model = Net()
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [6]:
def train_epoch(dataloader, model, loss_fn, optimizer):
    train_loss = 0
    num_batches = len(dataloader)

    for x, y in dataloader:
        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    
    return train_loss/num_batches

In [7]:
def test_epoch(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    
    return test_loss / num_batches, correct / size

In [8]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loss = train_epoch(loaded_train, model, loss_function, optimizer)
    print(f"Train Loss: {train_loss:>7f}")

    test_loss, accuracy = test_epoch(loaded_test, model, loss_function)
    print(f"Test Loss: {test_loss:>8f}, Accuracy: {(100*accuracy):>0.1f}% \n")

Epoch 1
-------------------------------
Train Loss: 2.175965
Test Loss: 2.166362, Accuracy: 90.5% 

Epoch 2
-------------------------------
Train Loss: 2.167199
Test Loss: 2.165099, Accuracy: 92.8% 

Epoch 3
-------------------------------
Train Loss: 2.166508
Test Loss: 2.164514, Accuracy: 94.1% 

Epoch 4
-------------------------------
Train Loss: 2.166001
Test Loss: 2.164100, Accuracy: 94.9% 

Epoch 5
-------------------------------
Train Loss: 2.165809
Test Loss: 2.164473, Accuracy: 95.3% 

